In [1]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Build a LangChain agent using Agent Platform

This notebook shows you how to set up a LangChain agent using Gemini Enterprise Agent Platform.

### Install Agent Platform SDK for Python

Install the latest version of the Agent Platform SDK for Python as well as extra dependencies for LangChain:

In [ ]:
!pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,langchain] cloudpickle==3.0.0 \
    "pydantic>=2.10" \
    requests

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [2]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>

<div class="alert alert-block alert-warning">
If prompted, in the <b>Kernel Restarting</b> dialog, click <b>Ok</b>.
</div>

### Set Google Cloud project information and initialize Vertex AI

Before running this cell to initialize Vertex AI, update the configuration values for the <b>PROJECT_ID, LOCATION, and STAGING_BUCKET.</b>

In [ ]:
PROJECT_ID = "YOUR_LAB_PROJECT_ID"
LOCATION = "YOUR_LAB_REGION"
STAGING_BUCKET = "YOUR_LAB_STAGING_BUCKET"

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

## Wikipedia Tool
In this section, you'll define multiple Python functions that access the Wikipedia API. Later we'll use these Python functions as Tools in our Reasoning Engine agent.

### Define functions for accessing the Wikipedia API

In [ ]:
import requests

# get full text of the wiki page
def get_wiki_full_text(wiki_title):
    # get the page
    url = f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts&titles={wiki_title}&explaintext=true&format=json"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
    }
    response = requests.get(url, headers=headers)
    data = response.json()

    # extract plain text
    page_id = next(iter(data["query"]["pages"]))
    plain_text = data["query"]["pages"][page_id]["extract"]
    return plain_text

### Define Tool function for the Wikipedia API

In [13]:
# make a query on Wikipedia for a topic
def query_with_wikipedia(
    query: str = "Fallingwater",
):
    """
    Finds answer for any topic or object from Wikipedia and returns a dictionary containing the answer.

    Args:
        query: the name of object or topic to find.

    Example: {"answer": "Fallingwater is a house designed by the architect Frank Lloyd Wright in 1935."}
    """

    wiki_full_text = get_wiki_full_text(query)  # calls Wikipedia API to get full text
    
    return {"answer": wiki_full_text}

In [ ]:
query_with_wikipedia()

## Agent Search Tool
In this section, you will define a tool to call the /ask_gms endpoint on the Cloud Run service to query the Google Merchandise Shop dataset using Agent Search. Before using this tool, you must deploy the Cloud Run service using the deploy script: /ag-web/app/deploy.sh.

### Define Tool funtion for Agent Search

The Cloud Run host name can be found on Console > Cloud Run > ag-web > URL at the top.

In [ ]:
endpoint = !gcloud run services describe ag-web --region=us-central1 --format="value(status.url)"

GOOGLE_SHOP_VERTEXAI_SEARCH_URL = (
    endpoint[0] + "/ask_gms"
)
print(GOOGLE_SHOP_VERTEXAI_SEARCH_URL)

In [ ]:
def find_product_from_googleshop(product_name: str, product_description: str):
    """
    Find a product with the product_name and product_description from
    Google Merch Shop and returns a dictionary containing product details.
    """

    params = {"query": product_name + " " + product_description}
    response = requests.get(
        GOOGLE_SHOP_VERTEXAI_SEARCH_URL, params
    )  # calls Vertex AI Search
    item = response.json()
    productDetails = f"""
        {item['title']} is a product sold at Google Merch Shop. The price is {item['price']}. 
        You can buy the product at their web site: {item['link']}.
    """

    return {"productDetails": productDetails}

## Define and deploy the Agent
In this section, you create a LangChain agent with the Wikipedia Tool and Search Tool, and deploy it to the Agent Runtime on Gemini Enterprise Agent Platform.

### Define the agent

In [ ]:
from vertexai import agent_engines

model_name = "GEMINI_MODEL_ID"  # please change
agent = agent_engines.LangchainAgent(
    model=model_name,
    tools=[query_with_wikipedia, find_product_from_googleshop],
    agent_executor_kwargs={"return_intermediate_steps": True},
)

### Test it locally

In [ ]:
# input_text = "Google Dino Pin. A collectible enamel pin featuring the iconic Lonely T-Rex dinosaur from the Google Chrome browser offline error page. Great product."
# input_text = "Google Bike Enamel Pin. This is a collectible lapel pin featuring a colorful bicycle with a basket, likely representing Google's company culture and its promotion of cycling as a sustainable mode of transportation."
# input_text = "Android Pen. This is a blue ballpoint pen featuring the Android logo. It appears to be a promotional or branded item associated with the Android operating system"
# input_text = "Google Chrome Dinosaur Game Pin. where can i buy it? Write the answer in plaintext."
input_text = (
    "Google Bike Enamel Pin. where can i buy it? Write the answer in plaintext."
)
agent.query(input=input_text)

### Deploy the agent to the Agent Runtime

In [ ]:
remote_agent = agent_engines.create(
    agent,
    requirements=[
        "google-cloud-aiplatform[agent_engines,langchain]",
        "cloudpickle==3.0.0",
        "pydantic>=2.10",
        "requests",
    ],
)

### Test it

In [ ]:
remote_agent.query(input="Where can I buy the Chrome Dino Rope Lanyard?")

In [ ]:
remote_agent.query(input="What is Fallingwater? Write the answer in plaintext.")